# Agentic AI and RAGs ?

Using only open source frameworks.
- Tiny KB with FAISS + FakeEmbeddings
- Free Wikipedia utility (no API keys)
- Rule-based planner
- Stub summarizer with optional tiny HF model (sshleifer/tiny-gpt2)
Run all cells top to bottom in Colab.

In [ ]:
!pip install -q langchain langchain-community faiss-cpu wikipedia transformers accelerate sentencepiece

## 1) Build the KB retriever

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FakeEmbeddings

kb_docs = [
    Document(
        page_content="""
Agentic systems reason step-by-step about which tools to call instead of invoking tools blindly.
Their core loop is: (1) interpret the user goal, (2) inspect available context, (3) decide whether tools are needed,
(4) call one or more tools in a planned sequence, and (5) synthesize an answer grounded in the tool results.

Key principles for agentic behavior:
- Prefer direct answering from retrieved or provided context when sufficient; avoid unnecessary tool calls.
- Choose the simplest tool or minimal set of tools that can satisfy the user’s goal.
- When multiple tools are needed, sketch a short plan (ordered tool calls with dependencies) before executing.
- After each tool call, update an internal scratchpad and re-evaluate whether more tools are still necessary.
- Handle tool failures gracefully: inspect error messages, adjust inputs, or fall back to alternative tools.

Agentic systems must also enforce safety and constraints:
- Do not call tools that violate policy or appear unsafe given the user’s intent.
- Throttle repetitive or redundant calls that would waste resources without improving the answer.
- Be explicit in reasoning about trade-offs (e.g., precision vs. latency) when picking tools.
""".strip(),
        metadata={"source": "kb:agentic_concept"},
    ),
    Document(
        page_content="""
Retrievers fetch grounding passages from a knowledge base and are the primary interface to internal documents.
Given a user query, the system should: (1) normalize and possibly expand the query, (2) retrieve top-k candidates,
(3) read them carefully, and (4) base the answer primarily on those passages.

Best practices for using retrievers:
- Start with a moderately small k (e.g., 3–10) to keep context focused; increase k only if evidence is clearly missing.
- Reformulate the query if results look noisy or off-topic (e.g., add constraints, synonyms, or key entities).
- Use metadata filters (dates, document types, domains) when available to narrow down to the most relevant sources.
- When multiple passages agree, highlight the shared core facts; when they conflict, surface the disagreement explicitly.

Citing retriever outputs:
- Quote or paraphrase the most important sentences and explicitly cite their source identifiers (e.g., document IDs).
- Avoid cherry-picking single passages that contradict the majority of evidence without acknowledging the discrepancy.
- If retrieved passages do not directly support a precise answer, say so instead of guessing, and suggest follow-up queries.
""".strip(),
        metadata={"source": "kb:retrievers"},
    ),
    Document(
        page_content="""
Wikipedia is a broad-coverage, free fallback when the curated knowledge base lacks coverage or appears incomplete.
It should not be the first resort when high-quality internal documents exist, but can complement them for general facts,
background context, or definitions.

Guidelines for using Wikipedia:
- First attempt retrieval over the internal KB; only query Wikipedia if internal retrieval yields little or no relevant evidence.
- Use Wikipedia mainly for widely accepted, non-controversial information (basic facts, dates, definitions, high-level overviews).
- For critical or sensitive topics (e.g., medical, legal, or contentious political claims), treat Wikipedia as a starting point,
  not as the final authority; cross-check with more authoritative sources when possible.
- Pay attention to section structure: lead sections usually contain stable, high-level facts; deeper sections may be more speculative.

Communicating Wikipedia usage to users:
- Clearly indicate when information is derived from Wikipedia or similar external encyclopedic sources.
- Prefer concise, well-supported statements over niche details that rely on poorly sourced or disputed claims.
- If a Wikipedia article appears inconsistent or low quality, either omit those details or explicitly flag the uncertainty.
""".strip(),
        metadata={"source": "kb:wikipedia_tip"},
    ),
    Document(
        page_content="""
When evidence is thin, ambiguous, or conflicting, the system must be transparent about uncertainty instead of fabricating detail.
Honesty requires separating what is directly supported by sources from what is inferred, and clearly expressing confidence levels.

Practical rules for honest behavior:
- Always distinguish between: (a) facts directly supported by retrieved passages, (b) reasonable inferences, and (c) speculation.
- If key information is missing, explicitly say that the available sources do not answer the question fully.
- Do not invent citations, URLs, names, or numbers to fill gaps; if you cannot support them, do not present them as factual.
- When sources conflict, summarize each position, highlight the disagreement, and avoid taking a definitive stance without justification.

Helpful ways to express uncertainty:
- Use phrases like “the retrieved sources indicate…”, “based on the available evidence…”, or “the documents do not specify…”.
- Suggest concrete next steps: clarifying the question, expanding or narrowing retrieval, or consulting domain experts or
  authoritative references.
- Prefer being conservatively accurate over confidently wrong, even if that means providing a partial answer.
""".strip(),
        metadata={"source": "kb:honesty"},
    ),
    Document(
        page_content="""
Answers should be concise, focused, and well-structured, typically within 2–4 sentences for straightforward questions.
Lead with the main conclusion, then briefly justify it using the most relevant evidence and clearly marked citations.

Style and structure guidelines:
- Directly answer the user’s question in the first sentence whenever possible.
- Avoid long preambles, restating the entire question, or narrating internal reasoning unless explicitly asked.
- Use clear, concrete language; minimize jargon and define necessary technical terms in simple words.
- For complex topics, provide a short high-level summary first, then optionally a brief, structured breakdown (e.g., bullet points).

Citation and formatting:
- Cite the minimal set of sources that substantively support the key claims, rather than listing all retrieved documents.
- Integrate citations inline, near the claims they support, to make it easy to trace evidence.
- Keep formatting clean: short paragraphs, occasional bullets for lists, and no unnecessary verbosity.

When to expand beyond 2–4 sentences:
- If the user explicitly asks for a detailed explanation, tutorial, or step-by-step reasoning.
- If the topic is safety-critical and requires more context, caveats, or assumptions to avoid misinterpretation.
In all cases, maintain clarity and relevance as the primary goals.
""".strip(),
        metadata={"source": "kb:style"},
    ),
]

embeddings = FakeEmbeddings(size=256)
vs = FAISS.from_documents(kb_docs, embeddings)
retriever = vs.as_retriever(search_kwargs={"k": 3})
print("KB ready with", len(kb_docs), "docs")

## 2) Open source external tool: Wikipedia search

In [ ]:
from langchain_community.utilities import WikipediaAPIWrapper

wiki = WikipediaAPIWrapper(lang='en', top_k_results=2, doc_content_chars_max=1000)

def wiki_search(query: str, k: int = 2):
    # Use WikipediaAPIWrapper to search for the query and return short snippets with titles
    try:
        docs = wiki.load(query)
        snippets = [
            {
                "title": d.metadata.get("title", "Unknown"),
                "summary": d.page_content[:500].strip(),
            }
            for d in docs[:k]
        ]
        return snippets, None
    except Exception as e:
        return [], str(e)

print(wiki_search('Python programming')[0][:1])


## 3) Simple planner (rule-based)

In [ ]:
kb_keywords = ['agentic', 'retriever', 'citation', 'ground', 'transparen', 'honest']

def plan(question: str):
    # Simple keyword-based planner: prefer the KB if the question mentions a known KB topic,
    # otherwise fall back to Wikipedia.
    q_lower = question.lower()
    matched = [kw for kw in kb_keywords if kw in q_lower]
    if matched:
        return {'action': 'kb', 'matched_keywords': matched, 'reason': 'question matches known KB topic(s)'}
    return {'action': 'wiki', 'matched_keywords': [], 'reason': 'no KB keyword matched, falling back to Wikipedia'}

print(plan('How to ground answers?'))
print(plan('Who created Python?'))


## 4) Answer function with stub or tiny HF model

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models.fake import FakeListChatModel
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

prompt = ChatPromptTemplate.from_template("You are a helpful agentic assistant. Use the given context and wiki snippets."
    "If there is little evidence, say so and suggest a follow-up query."
    "Cite sources like [kb:doc1] or [wiki:Title]."
    "Question: {question}"
    "Context:{context}"
    "Wiki:{wiki}"
    "Answer:"
)

def get_tiny_generator(model_id: str = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)
    return pipeline('text-generation', model=model, tokenizer=tok, torch_dtype=torch.float32, device_map='auto')

def summarize_with_tiny(prompt_text: str, max_new_tokens: int = 120):
    gen = get_tiny_generator()
    out = gen(
        prompt_text,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_return_sequences=1,
        pad_token_id=gen.tokenizer.eos_token_id,
    )
    full = out[0]["generated_text"]
    completion = full[len(prompt_text):].strip()
    return completion

def answer_question(question: str, use_tiny_model: bool = True):
    pl = plan(question)
    docs = retriever.invoke(question) if pl['action']=='kb' else []
    wiki_snips = []
    wiki_err = None
    if pl['action']=='wiki':
        wiki_snips, wiki_err = wiki_search(question)
    context_text = '\n'.join([f"[{d.metadata.get('source')}] {d.page_content}" for d in docs]) or 'No KB context.'
    wiki_text = '\n'.join([f"[wiki:{s['title']}] {s['summary']}" for s in wiki_snips]) or 'No wiki snippets.'
    messages = prompt.format_messages(question=question, context=context_text, wiki=wiki_text)
    if use_tiny_model:
        final_answer = summarize_with_tiny(messages[0].content)
    else:
        stub = FakeListChatModel(responses=['Stub summary based on provided context and wiki snippets.'])
        final_answer = stub.invoke(messages).content
    return {
        'plan': pl,
        'kb_sources': [d.metadata.get('source') for d in docs],
        'wiki_sources': [s.get('title') for s in wiki_snips],
        'wiki_error': wiki_err,
        'answer': final_answer,
    }

## 5) Quick check on sample questions

In [ ]:
tests = [
    "How should an agent decide whether to use a retriever?",  # covered by the KB
    "Who created the Python programming language?",  # needs external (Wikipedia) info
    "What is the capital of Wakanda?",  # ambiguous / no good evidence in either source
]

for q in tests:
    res = answer_question(q, use_tiny_model=True)
    print('Q:', q)
    print('Plan:', res['plan'])
    print('KB sources:', res['kb_sources'])
    print('Wiki sources:', res['wiki_sources'])
    print('Answer:', res['answer'])
    print('---')
